**`ingest_properties`**

Download and prepare property (non-spatial assessment-roll) datasets.

# Configure

In [ ]:
import argparse

from openplaces.io.ingester import Ingester

In [ ]:
# Define arguments
parser = argparse.ArgumentParser(description='Ingest properties using a recipe')
parser.add_argument(
    '--recipe_id',
    help='Identifier of the recipe (e.g., "US-FL_property-fldor-2026")',
)
parser.add_argument(
    '--admin_ids',
    help='Administrative unit IDs to ingest (e.g., "US-FL-AL")',
    nargs='*',
)
parser.add_argument(
    '--partition_ids',
    help='Partition IDs to ingest (e.g., a year: "2025")',
    nargs='*',
)
parser.add_argument(
    '--reprocess',
    help='Reprocess input data from downloaded file',
    action='store_true',
)
parser.add_argument(
    '--redownload',
    help='Redownload input data from original source',
    action='store_true',
)
parser.add_argument(
    '--keep_unzipped',
    help='If True, keeps unzipped datasets in heap folder after processing',
    action='store_true',
)
parser.add_argument(
    '--verbose',
    help='If True, print outputs while processing data',
    action='store_true',
)

# Test arguments

In [ ]:
ARGS_TEST = (
    # Florida Department of Revenue NAL assessment roll
    '--recipe_id US-FL_property-fldor-2026 '
    # Alachua, Brevard, and Duval counties, FL
    '--admin_ids US-FL-AL US-FL-BE US-FL-DU '
    # 2025 Final roll
    '--partition_ids 2025 '
    # '--reprocess '
    # '--redownload '
    '--verbose '
    # '--keep_unzipped '
)

# Convert argument string to list of strings
args_list = [x for x in ARGS_TEST.split(' ') if x != '']

# Parse list of arguments
args = parser.parse_args(args_list)

# Display arguments to check if parsing worked as expected
args

In [ ]:
# Show recipe parameters
from openplaces.recipe import get_recipe_by_id
from openplaces.utils import pretty_print

pretty_print(get_recipe_by_id(args.recipe_id))

# Ingest property data

In [ ]:
ingester = Ingester(
    args.recipe_id,
    args.admin_ids,
    partition_ids=args.partition_ids,
    verbose=args.verbose,
)

In [ ]:
ingester.ingest(
    reprocess=args.reprocess,
    redownload=args.redownload,
    keep_unzipped=args.keep_unzipped,
)

---
# Convert to script

*The above line and heading identify the end of the script.*

*Code below this marker will not be included in the converted `.py` script.*

In [ ]:
from openplaces.flow import convert_to_script

COMMIT = True

convert_to_script(commit=COMMIT)

# Test script

In [ ]:
# from openplaces.flow import test_script

# test_script(*args_list, committed=COMMIT)

# Inspect results

## Sample rows

In [ ]:
import pandas as pd

from openplaces import get_entities

df = get_entities(args.recipe_id, args.admin_ids, partition_id=args.partition_ids[0])
print(len(df))
pd.options.display.max_rows = max(len(df.columns), 60)
df.sample(5).T

## Value distributions

Market and land value are both heavily right-skewed, so plotted on a log10 x-axis.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, col in zip(axes, ['market_value', 'land_value'], strict=True):
    values = df[col].dropna()
    values = values[values > 0]
    ax.hist(np.log10(values), bins=50, color='#4c78a8', alpha=0.85)
    ax.set_xlabel(f'log10({col})')
    ax.set_ylabel('count')
    ax.set_title(col)
fig.tight_layout()

## Use-group breakdown by county

`get_entities` adds an `admin3_id` column automatically when combining
output from multiple administrative units.

In [ ]:
from openplaces.viz import plot_tabulation

_ = plot_tabulation(
    df,
    y_cat='admin3_id',
    x_cat='use_group_code',
    v='n',
    x_max_n=8,
    title='% of parcels by DOR use-group code, per county',
)

## Median values by county

In [ ]:
df.groupby('admin3_id', observed=True)[['market_value', 'land_value']].median()